In [ ]:
!pip install -q langgraph

In [ ]:
from IPython.display import Image
from enum import Enum
from langchain_core.runnables import Runnable
from langgraph.graph import END, START, StateGraph
from pathlib import Path
from typing import TypedDict


def display_graph(runnable: Runnable, output_png: Path):
    with output_png.open(mode="wb") as file:
        file.write(runnable.get_graph().draw_mermaid_png())

    display(Image(output_png, format="png"))

In [ ]:
class Category(Enum):
    BILLING = "billing"
    TECHNICAL = "tech"
    SPAM = "spam"

class Priority(Enum):
    LOW = 1
    NORMAL = 2
    HIGH = 3

class Ticket(TypedDict):
    text: str
    category: Category
    priority: Priority
    response: str

In [ ]:
def classify(state: Ticket) -> dict:
    text = state["text"].lower()
    if any(k in text for k in ["refund", "invoice", "charge", "payment"]):
        category = Category.BILLING
    elif any(k in text for k in ["crash", "bug", "error", "not working"]):
        category = Category.TECHNICAL
    else:
        category = Category.SPAM

    priority: Priority = Priority.HIGH if "urgent" in text or "!!!" in text else Priority.LOW
    print(f"[classify] -> {category} / {priority}")
    return {"category": category, "priority": priority}


def billing(state: Ticket) -> dict:
    return {"response": "Billing: we will review your invoice within 24h."}


def technical(state: Ticket) -> dict:
    return {"response": "Technical: please share the error log; we are on it."}


def reject(state: Ticket) -> dict:
    return {"response": "(auto-rejected as spam)"}


def human_review(state: Ticket) -> dict:
    escalated = state["response"] + " [ESCALATED TO HUMAN AGENT]"
    return {"response": escalated}

In [ ]:
def maybe_escalate(state: Ticket) -> str:
    if (state["priority"] == Priority.HIGH or state["priority"] == Priority.NORMAL) and not state["category"] == Category.SPAM:
        return "human_review"
    return END

In [ ]:
graph_builder = StateGraph(Ticket)
graph_builder.add_node("classify", classify)
graph_builder.add_node("billing", billing)
graph_builder.add_node("technical", technical)
graph_builder.add_node("reject", reject)
graph_builder.add_node("human_review", human_review)

# category -> processing_node
processing_nodes_map = { Category.BILLING: "billing", Category.TECHNICAL: "technical", Category.SPAM: "reject" }
processing_nodes_map = { key.value: value for key, value in processing_nodes_map.items() }

graph_builder.add_edge(START, "classify")
graph_builder.add_conditional_edges("classify", lambda x: x["category"].value, processing_nodes_map)

for processing_node in processing_nodes_map.values():
    graph_builder.add_conditional_edges(processing_node, maybe_escalate, ["human_review", END])

graph_builder.add_edge("human_review", END)

graph = graph_builder.compile()

In [ ]:
display_graph(graph, Path("/content/graph.png"))

In [ ]:
examples = [
    "My last invoice has a wrong charge, please refund.",
    "URGENT!!! the app crashes on startup, bug?",
    "Buy cheap watches now"
]

for example in examples:
    print(f">> {example}")

    result = graph.invoke({ "text": example })
    print(result["response"])
    print()